# Módulo de radiación cósmica
**NASA Space Apps Challenge 2026 – Monitoreo de salud para astronautas**

Este cuaderno simula la radiación que recibe un astronauta durante una misión, usando **tormentas solares reales** de la API DONKI de NASA.

Para ejecutarlo: correr las celdas en orden, de arriba hacia abajo (botón ▶ de cada celda o `Ctrl + Enter`).

## Antes de empezar: guardar la clave de NASA

La clave no se escribe en el código para que no quede a la vista si se comparte el cuaderno. Se guarda en los **Secretos** de Colab:

1. En la barra de la izquierda, hacer clic en el ícono de la **llave** 🔑.
2. Clic en **"Agregar nuevo secreto"**.
3. En *Nombre* poner: `NASA_API_KEY`
4. En *Valor* pegar la clave.
5. Activar el interruptor de **acceso del cuaderno**.

## Paso 1: Parámetros del escenario

Todos los números están juntos aquí para poder cambiarlos fácilmente cuando terminemos la investigación. Son **valores aproximados de ejemplo**.

In [ ]:
FECHA_INICIO = "2024-05-01"   # Periodo de la misión simulada
FECHA_FIN = "2024-05-31"      # (mayo 2024 tuvo tormentas solares muy fuertes)

DOSIS_FONDO = 1.8             # mSv por día en viaje a Marte (rayos cósmicos galácticos)
VARIACION_FONDO = 0.15        # variación aleatoria diaria (para que no sea una línea plana)

DOSIS_TORMENTA_DIA1 = 10.0    # mSv extra el día que empieza la tormenta (valor de ejemplo)
DOSIS_TORMENTA_DIA2 = 4.0     # mSv extra el día siguiente (la tormenta va bajando)
PROTECCION_REFUGIO = 0.70     # el refugio bloquea ~70 % de la dosis extra (supuesto)

LIMITE_CARRERA = 600.0        # mSv, límite de NASA para toda la carrera
DOSIS_PREVIA = 0.0            # mSv que el astronauta ya tenía de misiones anteriores

SEMILLA = 42                  # para que la simulación salga igual cada vez

## Paso 2: Obtener las tormentas solares reales de NASA

Se consulta la API **DONKI**, que registra los eventos de clima espacial. Se piden los eventos **SEP** (*Solar Energetic Particles*), que son las tormentas de partículas solares peligrosas para los astronautas. De cada evento solo se guarda el día.

In [ ]:
import json
import urllib.request
from datetime import date, timedelta

def leer_clave():
    """Lee la clave desde los Secretos de Colab. Si no está, usa DEMO_KEY
    (funciona, pero permite pocas consultas por hora)."""
    try:
        from google.colab import userdata
        return userdata.get("NASA_API_KEY")
    except Exception:
        print("Aviso: no se encontró el secreto NASA_API_KEY, se usa DEMO_KEY.")
        return "DEMO_KEY"

def obtener_tormentas(inicio, fin):
    url = ("https://api.nasa.gov/DONKI/SEP"
           f"?startDate={inicio}&endDate={fin}&api_key={leer_clave()}")
    try:
        with urllib.request.urlopen(url, timeout=30) as respuesta:
            eventos = json.loads(respuesta.read().decode("utf-8"))
    except Exception as error:
        print(f"No se pudo consultar a NASA ({error}). Se simula sin tormentas.")
        return []
    # Cada evento trae "eventTime", por ejemplo "2024-05-11T02:00Z"; nos quedamos con el día.
    fechas = sorted({e["eventTime"][:10] for e in (eventos or [])})
    print(f"Tormentas solares encontradas: {len(fechas)} -> {fechas}")
    return [date.fromisoformat(f) for f in fechas]

tormentas = obtener_tormentas(FECHA_INICIO, FECHA_FIN)

## Paso 3: Simular la dosis diaria

Para cada día:
- **Dosis de fondo:** el valor base más una pequeña variación aleatoria.
- **Pico de tormenta:** si hay tormenta ese día se suma una dosis extra grande, y al día siguiente una más pequeña.
- Se calculan dos versiones para comparar: **sin refugio** (el astronauta no hace caso a la alerta) y **con refugio** (sigue la recomendación de la app).

In [ ]:
import random
import pandas as pd

def simular_dosis(inicio, fin, tormentas):
    random.seed(SEMILLA)
    dia, ultimo = date.fromisoformat(inicio), date.fromisoformat(fin)
    filas = []
    while dia <= ultimo:
        fondo = DOSIS_FONDO + random.uniform(-VARIACION_FONDO, VARIACION_FONDO)
        extra = 0.0
        if dia in tormentas:
            extra = DOSIS_TORMENTA_DIA1
        elif (dia - timedelta(days=1)) in tormentas:
            extra = DOSIS_TORMENTA_DIA2
        filas.append({
            "fecha": dia,
            "tormenta": dia in tormentas,
            "dosis_sin_refugio": round(fondo + extra, 2),
            "dosis_con_refugio": round(fondo + extra * (1 - PROTECCION_REFUGIO), 2),
        })
        dia += timedelta(days=1)
    return pd.DataFrame(filas)

tabla = simular_dosis(FECHA_INICIO, FECHA_FIN, tormentas)
tabla.head(10)

## Paso 4: Dosis acumulada y alertas

Se suma la dosis día a día, se calcula qué porcentaje del límite de 600 mSv lleva el astronauta y se le asigna una alerta a cada día:

| Alerta | Cuándo |
|---|---|
| ROJO | Hay tormenta solar ese día |
| AMARILLO | La dosis del día es más de 1,5 veces la normal |
| AVISO | Se pasó el 50 % del límite de carrera |
| VERDE | Todo normal |

In [ ]:
def calcular_alertas(tabla):
    tabla["acumulada"] = (DOSIS_PREVIA + tabla["dosis_con_refugio"].cumsum()).round(2)
    tabla["pct_limite"] = (tabla["acumulada"] / LIMITE_CARRERA * 100).round(1)
    alertas = []
    for _, fila in tabla.iterrows():
        if fila["tormenta"]:
            alertas.append("ROJO: tormenta solar. Ir al refugio y evitar salidas fuera de la nave.")
        elif fila["dosis_sin_refugio"] > DOSIS_FONDO * 1.5:
            alertas.append("AMARILLO: radiación elevada. Limitar tiempo en zonas poco protegidas.")
        elif fila["pct_limite"] >= 50:
            alertas.append("AVISO: se superó el 50 % del límite de carrera.")
        else:
            alertas.append("VERDE: dosis normal.")
    tabla["alerta"] = alertas
    return tabla

tabla = calcular_alertas(tabla)
print(f"Total del periodo: {tabla['dosis_con_refugio'].sum():.1f} mSv con refugio, "
      f"{tabla['dosis_sin_refugio'].sum():.1f} mSv sin refugio.")
tabla[["fecha", "dosis_con_refugio", "acumulada", "pct_limite", "alerta"]]

## Gráfico

Arriba: la dosis de cada día. En rojo claro lo que habría recibido **sin refugio** y en azul lo que recibe **siguiendo la alerta**. Esa diferencia muestra que la app protege al astronauta.

Abajo: la dosis acumulada durante la misión.

In [ ]:
import matplotlib.pyplot as plt

fig, (arriba, abajo) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
arriba.bar(tabla["fecha"], tabla["dosis_sin_refugio"], color="#f4a3a3", label="Sin refugio")
arriba.bar(tabla["fecha"], tabla["dosis_con_refugio"], color="#3b6fb6", label="Con refugio (siguiendo la alerta)")
arriba.set_ylabel("Dosis diaria (mSv)")
arriba.set_title("Radiación diaria en la misión simulada")
arriba.legend()

abajo.plot(tabla["fecha"], tabla["acumulada"], color="#3b6fb6", linewidth=2)
abajo.set_ylabel("Dosis acumulada (mSv)")
abajo.set_title(f"Dosis acumulada (límite de carrera: {LIMITE_CARRERA:.0f} mSv)")

fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## Guardar y descargar los resultados

Esta celda guarda la tabla como CSV y la descarga a la computadora, para pasársela a Sebastián (búsqueda de patrones).

In [ ]:
tabla.to_csv("radiacion_simulada.csv", index=False)
from google.colab import files
files.download("radiacion_simulada.csv")